In [1]:
from IPython.display import display, HTML 
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
from urllib.request import urlopen
import math
import time
import requests
import pandas as pd

kospi, kosdak, divident = 'gsum', 'ksd_gsum', 'div'

C:\Users\Admin\anaconda3\lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (4.0.0)/charset_normalizer (2.0.4) doesn't match a supported version!
  warnings.warn(


In [3]:
def load_csv_file(str):
    date = time.strftime("%y%m%d", time.localtime())
    item_list = pd.read_csv(f'data/naver/fin_{str}_{date}.csv', encoding='cp949') 
    return item_list

## 네이버 금융/주식/시가총액 화면

In [4]:
def call_nav_fin_gsum(string):

    driver = webdriver.Chrome()
    if(string == 'gsum'):
        driver.get('https://finance.naver.com/sise/sise_market_sum.naver?sosok=0')
    elif(string == 'ksd_gsum'):
        driver.get('https://finance.naver.com/sise/sise_market_sum.naver?sosok=1')
    time.sleep(0.5) # 초기화면 접근

    no_tgl_bxs = len(driver.find_elements(By.CSS_SELECTOR, 'td > input[type="checkbox"]'))
    toggle_boxes = [f'option{i}' for i in range(1, no_tgl_bxs+1)] # 체크박스 이름 준비

    pageN = 10
    max_tgl = 6
    item_list = pd.DataFrame([])
    for page in range(1,pageN+1):
        page_item_list = []
        for sheet in range(0, math.ceil(no_tgl_bxs/max_tgl)):
            dflt_toggle_boxes = driver.find_elements(By.CSS_SELECTOR, 'td.choice > input[type="checkbox"]')
            [elem.send_keys(Keys.SPACE) for elem in dflt_toggle_boxes]
            time.sleep(0.5) # 디폴트 체크박스 toggle

            sheet_toggle_boxes = toggle_boxes[sheet*max_tgl:min((sheet+1)*max_tgl, no_tgl_bxs+1)]
            new_toggle_boxes = [select_box for select_box in sheet_toggle_boxes]
            [driver.find_element(By.ID, box_id).send_keys(Keys.SPACE) for box_id in new_toggle_boxes]
            time.sleep(0.5) # 시트별 새로운 체크박스 toggle
            driver.find_element(By.CSS_SELECTOR, 'div.item_btn > a').click()
            time.sleep(0.5) # 토글 체크후 submit버튼 클릭

            soup = BeautifulSoup(driver.page_source, 'html.parser')
            table_head = [e.text.strip() for e in soup.select('thead > tr >th')][:-1]
            table_head.insert(3, '전일비변동')
            items = soup.select('tbody > tr') # 테이블 헤드 수집
            sheet_item_list = []
            for idx, item in enumerate(items):
                if(item.select_one('td.no')):
                    no = item.select_one('td.no').text
                    title = item.select_one('a.tltle').text
                    price_vals = [e.text.strip() for e in item.select('td.number')]
                    if(price_vals[1].strip()=='보합0'):
                        price_vals[1]=['보합',0]
                    else:
                        price_vals[1] = [price_vals[1].split('\n')[0], price_vals[1].split('\n')[1].strip()]

                    price_vals.insert(0, no)
                    price_vals.insert(1, title)
                    temp = price_vals[3][1]
                    price_vals.insert(3,price_vals[3][0])
                    price_vals[4] = temp

                    sheet_item_list.append(price_vals) # 테이블 내용 수집 
            sheet_item_df = pd.DataFrame(sheet_item_list, columns=table_head)

            if sheet == 0:
                page_item_list = sheet_item_df
            elif sheet < round(no_tgl_bxs/max_tgl):
                page_item_list = pd.concat([page_item_list, sheet_item_df[table_head[7:13]]], axis=1)
            else: 
                page_item_list = pd.concat([page_item_list, sheet_item_df[table_head[7:9]]], axis=1)
                # 페이지 내 시트 축적
        item_list = pd.concat([item_list, page_item_list], axis=0) # 페이지 내용 축적

        page_div = driver.find_element(By.CSS_SELECTOR, 'table.Nnavi > tbody > tr')
        if(page<pageN): # 지정 마지막 페이지 초과(pageN+1) 이동 금지
            page_div.find_element(By.LINK_TEXT, str(page+1)).click()
            time.sleep(1.5) # next page로
    print(f'~~ 총 {page}페이지 {sheet}시트 데이터를 수집 하였습니다~~')

    item_list.to_csv(f'data/naver/fin_{string}_{time.strftime("%y%m%d", time.localtime())}.csv', index=False, encoding='cp949')
    item_list.sample(10)

## 네이버 배당 화면 접근

In [5]:
def call_nav_fin_divident(string):
    driver = webdriver.Chrome()
    driver.get('https://finance.naver.com/sise/dividend_list.naver')
    time.sleep(0.5)

    pageN = 10
    item_list = []
    for page in range(1,pageN+1):
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table_head = [e.text.strip() for e in soup.select('thead > tr > th')]
        table_head[9] = '과거3년배당금_Max'
        


        items = soup.select('table.type_1.tb_ty> tbody > tr')
        for idx, item in enumerate(items):
            if(item.text.strip()):
                price_vals = [e.text.strip().replace(',','').replace('-','0') for e in item.select('td')]
                price_vals.insert(9, max(price_vals[9:12]))

                item_list.append({table_head[i]:e for i, e in enumerate(price_vals)})

        # next page로
        page_div = driver.find_element(By.CSS_SELECTOR, 'table.Nnavi > tbody > tr')
        if(page<pageN): # 지정 마지막 페이지 초과(pageN+1) 이동 금지
            page_div.find_element(By.LINK_TEXT, str(page+1)).click()
            time.sleep(1.5)
            
    print(f'~~ 총 {page}페이지 데이터를 수집 하였습니다~~')
    df = pd.DataFrame(item_list)
    df.to_csv(f'data/naver/fin_{string}_{time.strftime("%y%m%d", time.localtime())}.csv', index=False, encoding='cp949')

In [6]:
call_nav_fin_gsum(kospi)
call_nav_fin_gsum(kosdak)
call_nav_fin_divident(divident)

kospi_item_list = load_csv_file(kospi)
kosdak_item_list = load_csv_file(kosdak)
divident_item_list = load_csv_file(divident)

kosdak_item_list.tail(3)

selected_divident_items = divident_item_list[(divident_item_list['수익률(%)']>=5.0) &
                                             (divident_item_list['ROE(%)']>=5.0) &
                                             (divident_item_list['PER(배)']<=10.0) &
                                             (divident_item_list['PBR(배)']<=1.0) &
                                             (divident_item_list['배당성향(%)']>=25.0)]
display(selected_divident_items)
p_kospi = kospi_item_list[kospi_item_list['전일비변동']=='하락'].sort_values('등락률', ascending=False)
p_kosdak = kosdak_item_list[kosdak_item_list['전일비변동']=='하락'].sort_values('등락률', ascending=False)
proposed_kospi = pd.merge(selected_divident_items, p_kospi, how='inner', on=['종목명'])
proposed_kosdak = pd.merge(selected_divident_items, p_kosdak, how='inner', on=['종목명'])

display(proposed_kospi.T)
display(proposed_kosdak.T)


~~ 총 10페이지 4시트 데이터를 수집 하였습니다~~
~~ 총 10페이지 4시트 데이터를 수집 하였습니다~~
~~ 총 10페이지 데이터를 수집 하였습니다~~


,종목명,현재가,기준월,배당금,수익률(%),배당성향(%),ROE(%),PER(배),PBR(배),과거3년배당금_Max,1년전,2년전,3년전
1,레드캡투어,11900,25.03,2150,18.07,177.53,9.64,7.26,0.67,450,450,450,400
13,정다운,2690,24.12,250,9.29,68.61,9.33,7.31,0.65,300,300,100,100
14,HB인베스트먼트,2205,24.12,200,9.07,89.81,8.28,6.99,0.49,0,0,0,0
19,현대해상,25350,24.03,2063,8.14,28.16,7.38,4.82,0.40,1965,1965,1480,1000
22,영보화학,4455,24.12,350,7.86,30.26,13.01,3.16,0.38,50,50,50,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,한일현대시멘트,17250,24.12,900,5.22,28.97,14.60,4.36,0.61,700,700,600,1200
156,YW,3840,24.12,200,5.21,37.63,5.31,9.62,0.36,200,200,150,100
158,코리안리,9990,25.04,515,5.16,28.74,9.44,4.89,0.41,458,458,311,324
167,대상우,17000,25.03,860,5.06,32.18,6.94,7.26,0.49,810,810,810,810


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
종목명,현대차우,현대차2우B,기아,강원랜드,LX인터내셔널,삼성화재우,이노션,현대차,삼성카드,동양생명,BNK금융지주,한일시멘트,삼성증권,코리안리
현재가_x,158300,159400,98200,18000,31000,321000,20000,206750,48500,7010,11750,18170,66900,9990
기준월,25.02,25.02,25.03,25.04,25.02,25.03,25.03,25.02,25.03,24.04,25.02,24.12,24.12,25.04
배당금,12050,12100,6500,1170,2000,19005,1175,12000,2800,400,650,1000,3500,515
수익률(%),7.61,7.59,6.62,6.5,6.45,5.92,5.88,5.8,5.77,5.71,5.53,5.5,5.23,5.16
배당성향(%),25.13,25.13,26.18,51.32,40.94,38.95,46.88,25.13,44.96,26.01,28.46,37.63,34.76,28.74
ROE(%),12.43,12.43,19.09,12.08,7.12,13.11,10.48,12.43,8.0,6.18,6.96,11.13,12.89,9.44
PER(배),4.6,4.6,4.12,7.48,5.97,8.74,7.74,4.6,6.88,3.05,4.56,5.46,4.32,4.89
PBR(배),0.51,0.51,0.71,0.82,0.37,0.98,0.77,0.51,0.5,0.25,0.31,0.58,0.53,0.41
과거3년배당금_Max,7050,7100,5600,930,3000,16005,900,7000,2500,620,625,800,3800,458


,0,1,2,3
종목명,노바텍,에스에이엠티,멀티캠퍼스,골프존
현재가_x,18250,2915,34350,66300
기준월,24.12,24.12,24.12,24.12
배당금,1409,200,2100,4000
수익률(%),7.72,6.86,6.11,6.03
배당성향(%),80.44,35.89,40.06,48.23
ROE(%),10.8,13.46,15.16,11.62
PER(배),8.95,4.79,5.7,8.24
PBR(배),0.87,0.59,0.82,0.89
과거3년배당금_Max,500,230,800,4500
